In [1]:
import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import joblib, os
from sklearn.preprocessing import StandardScaler

In [2]:
#importing both the files
og_df = pd.read_parquet(r"D:\Mortgage-Credit-Risk-Analytics\Data\processed\master_origination_clean.parquet")
m_df = pd.read_parquet(r"D:\Mortgage-Credit-Risk-Analytics\Data\processed\master_monthly_performance_clean.parquet",
                       columns=["loan_seq_no", "current_loan_delinquency_status",
                                                        "zero_balance_code"])   #only these 3 cols are required for the label

print("origination:", og_df.shape)
print("monthly    :", m_df.shape)
print("\norigination columns:")
print(og_df.columns)

origination: (550000, 33)
monthly    : (33135526, 3)

origination columns:
Index(['cred_score', 'first_payment_date', 'first_time_homebuyer_flag',
       'maturity_date', 'metro_code_msa', 'mortgage_insurance_percent',
       'unit_no', 'occupancy_status', 'og_cltv', 'og_dti', 'og_upb', 'og_ltv',
       'og_int_rate', 'channel', 'ppm_flag', 'amort_type', 'prop_state',
       'prop_type', 'postal_code', 'loan_seq_no', 'loan_purpose',
       'og_loan_term', 'no_of_borrowers', 'seller_name',
       'super_conforming_flag', 'pre_relief_refinance_loan_seq_no',
       'special_elig_program', 'relief_refinance_indicator', 'prop_val_method',
       'int_only_indicator', 'MI_cal_indicator', 'origination_year',
       'is_default'],
      dtype='str')


In [3]:
import pandas as pd

hpi = pd.read_excel(
    r"D:\Mortgage-Credit-Risk-Analytics\Data\raw\hpi_at_state.xlsx",
    skiprows=5)                          # header now on the right row


hpi = hpi[["Abbreviation", "Year", "HPI with 2000 base"]].copy()
hpi.columns = ["state", "year", "hpi"]

hpi["year"] = pd.to_numeric(hpi["year"], errors="coerce")
hpi["hpi"]  = pd.to_numeric(hpi["hpi"], errors="coerce")
hpi = hpi.dropna(subset=["state", "year", "hpi"])
hpi["year"] = hpi["year"].astype(int)

print(hpi.shape)
print("years:", hpi["year"].min(), "-", hpi["year"].max())
print("states:", hpi["state"].nunique())
print(hpi[hpi["state"] == "CA"].sort_values("year").iloc[-15:])

(2601, 3)
years: 1975 - 2025
states: 51
    state  year     hpi
240    CA  2011  135.20
241    CA  2012  134.84
242    CA  2013  149.00
243    CA  2014  169.96
244    CA  2015  179.79
245    CA  2016  193.75
246    CA  2017  206.23
247    CA  2018  219.53
248    CA  2019  226.66
249    CA  2020  231.85
250    CA  2021  261.33
251    CA  2022  302.76
252    CA  2023  308.25
253    CA  2024  321.70
254    CA  2025  324.12


In [4]:
print(hpi[(hpi["state"] == "CA") & (hpi["year"].between(2000, 2012))].to_string(index=False))


state  year    hpi
   CA  2000 100.00
   CA  2001 112.07
   CA  2002 123.86
   CA  2003 136.13
   CA  2004 164.52
   CA  2005 207.25
   CA  2006 228.94
   CA  2007 218.54
   CA  2008 175.35
   CA  2009 148.70
   CA  2010 143.74
   CA  2011 135.20
   CA  2012 134.84


In [5]:
hpi_at = hpi.rename(columns={"year": "origination_year", "hpi": "hpi_at_orig"})
hpi_lag = hpi.copy()
hpi_lag["origination_year"] = hpi_lag["year"] + 3
hpi_lag = hpi_lag.rename(columns={"hpi": "hpi_3yr_before"})[["state", "origination_year", "hpi_3yr_before"]]
macro = hpi_at.merge(hpi_lag, on=["state", "origination_year"], how="left")
macro["hpi_change_3yr_prior"] = (macro["hpi_at_orig"] / macro["hpi_3yr_before"] - 1) * 100
macro = macro[["state", "origination_year", "hpi_at_orig", "hpi_change_3yr_prior"]]

In [6]:
#Basel default label: ever 90+ DPD OR terminal credit event {2,3,9}
dpd = pd.to_numeric(m_df["current_loan_delinquency_status"], errors="coerce")
m_df["bad"] = dpd.ge(3) | m_df["zero_balance_code"].isin([2, 3, 9])
label = m_df.groupby("loan_seq_no")["bad"].max().astype(int)

og_df["is_default"] = og_df["loan_seq_no"].map(label).fillna(0).astype(int)

del m_df  #not required further

print(f"default rate:  {round(og_df["is_default"].mean() * 100, 2)}%")

default rate:  7.17%


In [7]:
# drop dead columns: constants, IDs, raw dates
DROP = ["amort_type", "special_elig_program", "prop_val_method",
        "int_only_indicator", "MI_cal_indicator",           # constants
        "postal_code", "pre_relief_refinance_loan_seq_no",  # IDs
        "first_payment_date", "maturity_date"]              # raw dates

og_df = og_df.drop(columns=DROP)

print("shape after drops:", og_df.shape)
print("\nremaining columns:")
print(og_df.columns)

shape after drops: (550000, 24)

remaining columns:
Index(['cred_score', 'first_time_homebuyer_flag', 'metro_code_msa',
       'mortgage_insurance_percent', 'unit_no', 'occupancy_status', 'og_cltv',
       'og_dti', 'og_upb', 'og_ltv', 'og_int_rate', 'channel', 'ppm_flag',
       'prop_state', 'prop_type', 'loan_seq_no', 'loan_purpose',
       'og_loan_term', 'no_of_borrowers', 'seller_name',
       'super_conforming_flag', 'relief_refinance_indicator',
       'origination_year', 'is_default'],
      dtype='str')


In [8]:
og_df = og_df.merge(macro, left_on=["prop_state", "origination_year"],
                    right_on=["state", "origination_year"], how="left").drop(columns="state")
print("macro coverage:", (1 - og_df["hpi_change_3yr_prior"].isna().mean())*100, "% of loans matched")

macro coverage: 99.69381818181819 % of loans matched


In [9]:
miss = og_df[og_df["hpi_change_3yr_prior"].isna()]
print("missing rows:", len(miss))
print("\nby state:")
print(miss["prop_state"].value_counts().head(10))
print("\nby year:")
print(miss["origination_year"].value_counts().sort_index())

missing rows: 1684

by state:
prop_state
PR    1538
GU     127
VI      19
Name: count, dtype: int64

by year:
origination_year
2000    184
2001    118
2002     98
2003    142
2004    207
2005    209
2006    193
2007    228
2008    164
2009     94
2010     47
Name: count, dtype: int64


In [10]:
#out-of-time split
train_df = og_df[og_df["origination_year"].between(2000, 2006)].copy()
val_df   = og_df[og_df["origination_year"] == 2007].copy()
test_df  = og_df[og_df["origination_year"].between(2008, 2010)].copy()

for name, d in [("train (00-06)", train_df), ("val (2007)", val_df), ("test (08-10)", test_df)]:
    print(f"{name:14} n={len(d):>7,}   default rate={d['is_default'].mean()*100:5.2f}%")

train (00-06)  n=350,000   default rate= 6.66%
val (2007)     n= 50,000   default rate=16.48%
test (08-10)   n=150,000   default rate= 5.25%


In [11]:
def clean_sentinels(d):
    d = d.copy()
    # numeric placeholders -> NaN
    d["cred_score"] = d["cred_score"].where(d["cred_score"] <= 850)
    d["og_ltv"]     = d["og_ltv"].where(d["og_ltv"] <= 105)
    d["og_cltv"]    = d["og_cltv"].where(d["og_cltv"] <= 200)
    d["og_dti"]     = d["og_dti"].where(d["og_dti"] < 999)
    d["mortgage_insurance_percent"] = d["mortgage_insurance_percent"].where(d["mortgage_insurance_percent"] < 999)
    d["unit_no"]         = d["unit_no"].where(d["unit_no"] < 90)
    d["no_of_borrowers"] = d["no_of_borrowers"].where(d["no_of_borrowers"] < 90)
    # categorical placeholders -> NaN
    d["first_time_homebuyer_flag"] = d["first_time_homebuyer_flag"].replace("9", np.nan)
    d["channel"]   = d["channel"].replace("9", np.nan)
    d["prop_type"] = d["prop_type"].replace("99", np.nan)
    return d

train_df = clean_sentinels(train_df)
val_df   = clean_sentinels(val_df)
test_df  = clean_sentinels(test_df)

# checking how much is now missing (train), so we know where imputation is required
print((train_df.isna().mean() * 100).round(2).sort_values(ascending=False).head(12))

og_dti                        2.21
cred_score                    0.43
hpi_at_orig                   0.33
hpi_change_3yr_prior          0.33
first_time_homebuyer_flag     0.06
no_of_borrowers               0.03
prop_type                     0.02
og_cltv                       0.01
og_ltv                        0.01
channel                       0.01
og_upb                        0.00
mortgage_insurance_percent    0.00
dtype: float64


In [12]:
for d in (train_df, val_df, test_df):
    d["msa_missing"] = (d["metro_code_msa"].astype(str) == "Unknown").astype(int)
    d["dti_missing"] = d["og_dti"].isna().astype(int)

In [13]:
# median imputation: medians learned on TRAIN ONLY, applied to all splits
IMPUTE = ["cred_score", "og_ltv", "og_cltv", "og_dti",
          "mortgage_insurance_percent", "unit_no", "no_of_borrowers", "hpi_at_orig", "hpi_change_3yr_prior"]
train_medians = train_df[IMPUTE].median()         

for d in (train_df, val_df, test_df):
    d[IMPUTE] = d[IMPUTE].fillna(train_medians)     

print("train medians used:")
print(train_medians)
print("\nremaining numeric NaNs after impute (train):", train_df[IMPUTE].isna().sum().sum())

train medians used:
cred_score                    731.000000
og_ltv                         75.000000
og_cltv                        75.000000
og_dti                         33.000000
mortgage_insurance_percent      0.000000
unit_no                         1.000000
no_of_borrowers                 2.000000
hpi_at_orig                   115.780000
hpi_change_3yr_prior           18.626297
dtype: float64

remaining numeric NaNs after impute (train): 0


In [14]:
CAT_COLS = ["loan_purpose", "occupancy_status", "prop_type", "first_time_homebuyer_flag",
            "channel", "ppm_flag", "prop_state", "super_conforming_flag",
            "relief_refinance_indicator", "seller_name"]

for d in (train_df, val_df, test_df):
    for c in CAT_COLS:
        d[c] = d[c].astype(str).replace("nan", "Missing").fillna("Missing")

print("categorical NaNs remaining (train):",
      train_df[CAT_COLS].isna().sum().sum())

categorical NaNs remaining (train): 0


In [15]:
for c in ["cred_score", "og_dti", "og_ltv", "og_cltv", "mortgage_insurance_percent",
          "metro_code_msa", "first_time_homebuyer_flag", "channel", "prop_type"]:
    print(f"{c:28} dtype={str(train_df[c].dtype):10} sample={train_df[c].unique()[:6]}")

cred_score                   dtype=float64    sample=[790. 771. 762. 737. 594. 763.]
og_dti                       dtype=float64    sample=[29. 30. 23. 35. 24. 42.]
og_ltv                       dtype=float64    sample=[92. 61. 76. 87. 80. 95.]
og_cltv                      dtype=float64    sample=[92. 61. 76. 87. 80. 95.]
mortgage_insurance_percent   dtype=float64    sample=[30.  0. 25. 12. 18. 20.]
metro_code_msa               dtype=string     sample=<ArrowStringArray>
['Unknown', '39300.0', '16974.0', '23460.0', '22744.0', '15260.0']
Length: 6, dtype: string
first_time_homebuyer_flag    dtype=str        sample=<ArrowStringArray>
['N', 'Y', 'Missing']
Length: 3, dtype: str
channel                      dtype=str        sample=<ArrowStringArray>
['R', 'T', 'Missing', 'C', 'B']
Length: 5, dtype: str
prop_type                    dtype=str        sample=<ArrowStringArray>
['SF', 'CO', 'PU', 'MH', 'CP', 'Missing']
Length: 6, dtype: str


In [16]:
# vintage-median rate learned on TRAIN only (for the adverse-pricing feature)
vintage_rate = train_df.groupby("origination_year")["og_int_rate"].median()
global_rate  = train_df["og_int_rate"].median()

# MSA frequency map learned on TRAIN only
msa_freq = train_df["metro_code_msa"].value_counts(normalize=True)

def add_features(d):
    d = d.copy()
    # 1. second-lien burden: how much of the combined LTV is a piggyback loan
    d["cltv_minus_ltv"] = (d["og_cltv"] - d["og_ltv"]).clip(lower=0)

    # 2. adverse pricing: rate paid vs the typical rate for that vintage
    d["rate_spread_vintage"] = d["og_int_rate"] - d["origination_year"].map(vintage_rate).fillna(global_rate)
    
    # 3. dollar payment obligation (annuity formula): size + rate + term in one number
    r = d["og_int_rate"] / 100 / 12
    n = d["og_loan_term"].clip(lower=1)
    d["est_monthly_pi"] = d["og_upb"] * r / (1 - (1 + r) ** (-n))

    # 4. MSA frequency encoding (456 levels= one number; unseen/Unknown= 0)
    d["msa_freq"] = d["metro_code_msa"].map(msa_freq).fillna(0)

    # 5. combined-risk: high leverage AND high payment burden together (compounding risk)
    d["ltv_dti_stress"] = ((d["og_ltv"] > 80) & (d["og_dti"] > 40)).astype(int)

    return d

train_df = add_features(train_df)
val_df   = add_features(val_df)
test_df  = add_features(test_df)

print(train_df[["cltv_minus_ltv", "rate_spread_vintage", "est_monthly_pi", "msa_freq", "ltv_dti_stress", "hpi_at_orig", "hpi_change_3yr_prior"]].describe(include="all").round(3))

       cltv_minus_ltv  rate_spread_vintage  est_monthly_pi  msa_freq  \
count      350000.000           350000.000      350000.000  350000.0   
mean            1.193                0.002        1027.536     0.147   
std             4.430                0.491         512.017     0.181   
min             0.000               -4.250          53.960       0.0   
25%             0.000               -0.250         638.061     0.003   
50%             0.000                0.000         928.714      0.01   
75%             0.000                0.250        1323.792     0.379   
max           100.000                3.825        5430.118     0.379   

       ltv_dti_stress  hpi_at_orig  hpi_change_3yr_prior  
count      350000.000   350000.000            350000.000  
mean            0.052      125.588                24.000  
std             0.223       28.381                15.364  
min             0.000      100.000                 1.802  
25%             0.000      106.890                12.468

In [17]:
for d in (train_df, val_df, test_df):
    d.drop(columns=["og_cltv"], inplace=True) #hihgly correlated with og_ltv and getting all the info of this through cltv_minus_ltv.

In [18]:
def fit_woe(train, feature, is_numeric, bins=10):
    x = train[[feature, "is_default"]].copy()
    if is_numeric:
        edges = np.quantile(x[feature].dropna(), np.linspace(0, 1, bins + 1))
        edges = np.unique(edges); edges[0], edges[-1] = -np.inf, np.inf
        x["bin"] = pd.cut(x[feature], edges)
    else:
        edges = None
        x["bin"] = x[feature].astype(str)
    g = x.groupby("bin", observed=True)["is_default"].agg(["count", "sum"])
    g["good"] = (g["count"] - g["sum"]) + 0.5
    g["bad"]  = g["sum"] + 0.5
    g["woe"]  = np.log((g["good"]/g["good"].sum()) / (g["bad"]/g["bad"].sum()))
    iv = (((g["good"]/g["good"].sum()) - (g["bad"]/g["bad"].sum())) * g["woe"]).sum()
    gwoe = np.log((g["good"].sum()/(g["good"].sum()+g["bad"].sum())) /
                  (g["bad"].sum()/(g["good"].sum()+g["bad"].sum())))
    # key numeric bins by their string label so fit and apply always match
    wmap = {str(k): v for k, v in g["woe"].items()}
    return wmap, gwoe, edges, iv

#what gets WoE-encoded
NUM_WOE = ["cred_score", "og_ltv", "og_dti", "og_int_rate", "og_upb", "og_loan_term",
           "mortgage_insurance_percent", "cltv_minus_ltv", "rate_spread_vintage",
           "est_monthly_pi", "msa_freq", "hpi_at_orig", "hpi_change_3yr_prior"]

CAT_WOE = ["loan_purpose", "occupancy_status", "prop_type", "first_time_homebuyer_flag",
           "channel", "prop_state", "ppm_flag", "super_conforming_flag",
           "relief_refinance_indicator", "seller_name", "ltv_dti_stress", "no_of_borrowers", "unit_no"]

FLAGS = ["msa_missing", "dti_missing"]   # already 0/1, pass through as-is

print("numeric to WoE :", len(NUM_WOE))
print("categorical to WoE:", len(CAT_WOE))
print("flags (pass-through):", FLAGS)

numeric to WoE : 13
categorical to WoE: 13
flags (pass-through): ['msa_missing', 'dti_missing']


In [19]:
woe_maps = {}     # feature -> (woe dict, global_woe, kind)
iv_rows  = []

# fit WoE + IV on TRAIN for every feature
for f in NUM_WOE:
    wmap, gwoe, edges, iv = fit_woe(train_df, f, True)
    woe_maps[f] = (wmap, gwoe, edges, "num"); iv_rows.append((f, iv, "numeric"))
for f in CAT_WOE:
    wmap, gwoe, edges, iv = fit_woe(train_df, f, False)
    woe_maps[f] = (wmap, gwoe, edges, "cat");  iv_rows.append((f, iv, "categorical"))

# the IV ranking table
iv_df = pd.DataFrame(iv_rows, columns=["feature", "IV", "type"]).sort_values("IV", ascending=False)
def band(v):
    return ("useless"   if v < 0.02 else "weak"   if v < 0.1 else "medium"
            if v < 0.3 else "strong" if v < 0.5 else "very strong (check leakage)")
iv_df["band"] = iv_df["IV"].apply(band)
print(iv_df.to_string(index=False))

                   feature       IV        type                        band
                cred_score 0.649105     numeric very strong (check leakage)
               hpi_at_orig 0.300370     numeric                      strong
               seller_name 0.267739 categorical                      medium
                    og_ltv 0.232069     numeric                      medium
      hpi_change_3yr_prior 0.215620     numeric                      medium
              og_loan_term 0.210887     numeric                      medium
       rate_spread_vintage 0.182674     numeric                      medium
                    og_dti 0.162119     numeric                      medium
           no_of_borrowers 0.139794 categorical                      medium
                  msa_freq 0.130095     numeric                      medium
                prop_state 0.114071 categorical                      medium
               og_int_rate 0.104677     numeric                      medium
            

In [20]:
LEAKAGE_DROP = ["seller_name"] # origination-time but vintage/underwriting proxy
selected = [f for f in iv_df.loc[iv_df["IV"] >= 0.02, "feature"] if f not in LEAKAGE_DROP]

def apply_woe(df, feature):
    wmap, gwoe, edges, kind = woe_maps[feature]
    key = pd.cut(df[feature], edges).astype(str) if kind == "num" else df[feature].astype(str)
    return key.map(wmap).astype(float).fillna(gwoe)

def build_matrix(df):
    out = pd.DataFrame(index=df.index)
    for f in selected:
        out[f + "_woe"] = apply_woe(df, f)
    for fl in FLAGS:
        out[fl] = df[fl].values
    out["is_default"] = df["is_default"].values
    return out

train_woe, val_woe, test_woe = build_matrix(train_df), build_matrix(val_df), build_matrix(test_df)
print("shapes:", train_woe.shape, val_woe.shape, test_woe.shape)
print("NaN:", train_woe.isna().sum().sum(), val_woe.isna().sum().sum(), test_woe.isna().sum().sum())
print(train_woe.head(3).round(3))

shapes: (350000, 18) (50000, 18) (150000, 18)
NaN: 0 0 0
   cred_score_woe  hpi_at_orig_woe  og_ltv_woe  hpi_change_3yr_prior_woe  \
0           1.332            0.634      -0.696                     0.312   
1           1.132            0.634       0.415                     0.195   
2           0.737            0.634      -0.085                     0.142   

   og_loan_term_woe  rate_spread_vintage_woe  og_dti_woe  no_of_borrowers_woe  \
0            -0.241                    0.992       0.197                0.346   
1             0.965                    0.006       0.197               -0.409   
2            -0.241                    0.992       0.609                0.346   

   msa_freq_woe  prop_state_woe  og_int_rate_woe  ltv_dti_stress_woe  \
0         0.531           0.394           -0.310               0.079   
1         0.531           0.076            0.282               0.079   
2        -0.144          -0.085            0.025               0.079   

   mortgage_insurance_pe

In [21]:
X = train_woe.drop(columns=["is_default"])
Xc = add_constant(X)
vif = pd.DataFrame({
    "feature": Xc.columns,
    "VIF": [variance_inflation_factor(Xc.values, i) for i in range(Xc.shape[1])]
})
vif = vif[vif["feature"] != "const"].sort_values("VIF", ascending=False)
print(vif.to_string(index=False))

                       feature       VIF
                   msa_missing 22.601141
                  msa_freq_woe 21.664778
              og_loan_term_woe  2.161915
               hpi_at_orig_woe  1.728364
      hpi_change_3yr_prior_woe  1.608421
       rate_spread_vintage_woe  1.493328
                    og_ltv_woe  1.389799
            ltv_dti_stress_woe  1.268606
               og_int_rate_woe  1.242115
mortgage_insurance_percent_woe  1.225467
                    og_dti_woe  1.171694
                cred_score_woe  1.116411
              loan_purpose_woe  1.069191
                prop_state_woe  1.042413
                   channel_woe  1.037190
           no_of_borrowers_woe  1.029960
                   dti_missing  1.005501


In [22]:
#since, msa_missing and msa_fre_woe are highly correlated, it is better to drop one of these.
# since msa_missing isnt that important its better to drop it.
for d in (train_woe, val_woe, test_woe):
    d.drop(columns="msa_missing", inplace=True, errors="ignore")

In [23]:
OUT = r"D:\Mortgage-Credit-Risk-Analytics\Data\features"
os.makedirs(OUT, exist_ok=True)

# transformed matrices (parquet: smaller, faster, preserves dtypes)
train_woe.to_parquet(os.path.join(OUT, "train_woe.parquet"), index=False)
val_woe.to_parquet(os.path.join(OUT, "val_woe.parquet"), index=False)
test_woe.to_parquet(os.path.join(OUT, "test_woe.parquet"), index=False)
iv_df.to_parquet(os.path.join(OUT, "iv_summary.parquet"), index=False)

# the transform recipe — so modelling reloads and applies the identical mapping (no refit, no leakage)
joblib.dump({
    "woe_maps": woe_maps,                    # per-feature WoE dict + global fallback + bin edges
    "selected": selected,
    "flags": FLAGS,
    "train_medians": train_medians.to_dict(),
    "vintage_rate": vintage_rate.to_dict(),
    "global_rate": float(global_rate),
    "msa_freq": msa_freq.to_dict(),
    "credit_event_codes": [2, 3, 9],
}, os.path.join(OUT, "woe_pipeline.pkl"))

print("saved WoE matrices + pipeline to:", OUT)
print("final feature count:", train_woe.shape[1] - 1)

saved WoE matrices + pipeline to: D:\Mortgage-Credit-Risk-Analytics\Data\features
final feature count: 16


In [24]:
X = add_constant(train_woe.drop(columns="is_default"))
vif = pd.DataFrame({"feature": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]})
print(vif[vif["feature"] != "const"].sort_values("VIF", ascending=False).to_string(index=False))
print("\nfinal WoE features:", [c for c in train_woe.columns if c != "is_default"])
print("shapes:", train_woe.shape, val_woe.shape, test_woe.shape)

                       feature      VIF
              og_loan_term_woe 2.091017
               hpi_at_orig_woe 1.722211
                  msa_freq_woe 1.664162
      hpi_change_3yr_prior_woe 1.608419
       rate_spread_vintage_woe 1.493196
                    og_ltv_woe 1.389111
            ltv_dti_stress_woe 1.268459
               og_int_rate_woe 1.241626
mortgage_insurance_percent_woe 1.225234
                    og_dti_woe 1.171644
                cred_score_woe 1.116402
              loan_purpose_woe 1.069163
                prop_state_woe 1.036983
                   channel_woe 1.035616
           no_of_borrowers_woe 1.029461
                   dti_missing 1.005498

final WoE features: ['cred_score_woe', 'hpi_at_orig_woe', 'og_ltv_woe', 'hpi_change_3yr_prior_woe', 'og_loan_term_woe', 'rate_spread_vintage_woe', 'og_dti_woe', 'no_of_borrowers_woe', 'msa_freq_woe', 'prop_state_woe', 'og_int_rate_woe', 'ltv_dti_stress_woe', 'mortgage_insurance_percent_woe', 'loan_purpose_woe', 'chann

In [25]:
# raw feature groups
RAW_NUM = ["cred_score", "og_ltv", "og_dti", "og_int_rate", "og_upb", "og_loan_term",
           "mortgage_insurance_percent", "no_of_borrowers", "rate_spread_vintage",
           "ltv_dti_stress", "msa_freq", 
           "hpi_at_orig", "hpi_change_3yr_prior"]                       # numerics + the surviving engineered flags
RAW_OHE = ["loan_purpose", "occupancy_status", "prop_type",
           "first_time_homebuyer_flag", "channel"]            # small categoricals -> one-hot
RAW_FREQ = ["prop_state"]                                      # 54 levels -> frequency-encode

# frequency map for prop_state, learned on TRAIN
state_freq = train_df["prop_state"].value_counts(normalize=True)

# one-hot categories are fixed from TRAIN so val/test get identical columns
ohe_categories = {c: sorted(train_df[c].dropna().unique()) for c in RAW_OHE}

# scaler fit on TRAIN numerics only
scaler = StandardScaler().fit(train_df[RAW_NUM])

def build_raw(df):
    out = pd.DataFrame(index=df.index)
    # 1. scaled numerics
    out[RAW_NUM] = scaler.transform(df[RAW_NUM])
    # 2. one-hot the small categoricals, using train's category list
    for c in RAW_OHE:
        for cat in ohe_categories[c]:
            out[f"{c}_{cat}"] = (df[c].astype(str) == str(cat)).astype(int)
    # 3. frequency-encode prop_state (unseen states= 0)
    out["prop_state_freq"] = df["prop_state"].map(state_freq).fillna(0)
    out["is_default"] = df["is_default"].values
    return out

train_raw, val_raw, test_raw = build_raw(train_df), build_raw(val_df), build_raw(test_df)
print("shapes:", train_raw.shape, val_raw.shape, test_raw.shape)
print("NaN:", train_raw.isna().sum().sum(), val_raw.isna().sum().sum(), test_raw.isna().sum().sum())
print("columns:", train_raw.shape[1] - 1, "features")
print(train_raw.columns.tolist())

shapes: (350000, 35) (50000, 35) (150000, 35)
NaN: 0 0 0
columns: 34 features
['cred_score', 'og_ltv', 'og_dti', 'og_int_rate', 'og_upb', 'og_loan_term', 'mortgage_insurance_percent', 'no_of_borrowers', 'rate_spread_vintage', 'ltv_dti_stress', 'msa_freq', 'hpi_at_orig', 'hpi_change_3yr_prior', 'loan_purpose_C', 'loan_purpose_N', 'loan_purpose_P', 'occupancy_status_I', 'occupancy_status_P', 'occupancy_status_S', 'prop_type_CO', 'prop_type_CP', 'prop_type_MH', 'prop_type_Missing', 'prop_type_PU', 'prop_type_SF', 'first_time_homebuyer_flag_Missing', 'first_time_homebuyer_flag_N', 'first_time_homebuyer_flag_Y', 'channel_B', 'channel_C', 'channel_Missing', 'channel_R', 'channel_T', 'prop_state_freq', 'is_default']


In [26]:
# drop one reference dummy per categorical (avoids the dummy-variable trap)
REF_DROP = ["loan_purpose_P", "occupancy_status_P", "prop_type_SF",
            "first_time_homebuyer_flag_N", "channel_R"]      # most common level of each = reference

for d in (train_raw, val_raw, test_raw):
    d.drop(columns=REF_DROP, inplace=True, errors="ignore")

# raw VIF (all numeric now, references dropped)
X = add_constant(train_raw.drop(columns="is_default"))
vif = pd.DataFrame({"feature": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]})
print(vif[vif["feature"] != "const"].sort_values("VIF", ascending=False).to_string(index=False))
print("\nraw features after reference-drop:", train_raw.shape[1] - 1)

                          feature      VIF
                      hpi_at_orig 5.491229
             hpi_change_3yr_prior 5.194335
                     og_loan_term 2.337900
                      og_int_rate 2.243067
       mortgage_insurance_percent 1.865773
                         msa_freq 1.862565
              rate_spread_vintage 1.812159
                   loan_purpose_N 1.750261
                   loan_purpose_C 1.738488
                           og_ltv 1.677578
                   ltv_dti_stress 1.512222
                           og_upb 1.432250
                  prop_state_freq 1.347849
      first_time_homebuyer_flag_Y 1.230281
                           og_dti 1.207074
                       cred_score 1.143851
                  no_of_borrowers 1.095270
                     prop_type_CO 1.080073
               occupancy_status_S 1.067478
               occupancy_status_I 1.067315
                        channel_T 1.056333
                     prop_type_PU 1.056327
           

In [27]:
OUT = r"D:\Mortgage-Credit-Risk-Analytics\Data\features"

train_raw.to_parquet(os.path.join(OUT, "train_raw.parquet"), index=False)
val_raw.to_parquet(os.path.join(OUT, "val_raw.parquet"), index=False)
test_raw.to_parquet(os.path.join(OUT, "test_raw.parquet"), index=False)

joblib.dump({
    "scaler": scaler,                      # fit on train numerics
    "raw_num": RAW_NUM,
    "ohe_categories": ohe_categories,      # frozen train category lists (column alignment)
    "ref_drop": REF_DROP,                  # reference dummies removed
    "state_freq": state_freq.to_dict(),    # prop_state frequency map
    "vintage_rate": vintage_rate.to_dict(),
    "global_rate": float(global_rate),
    "msa_freq": msa_freq.to_dict(),
}, os.path.join(OUT, "raw_pipeline.pkl"))

print("saved raw matrices + pipeline to:", OUT)
print("raw features:", train_raw.shape[1] - 1)

saved raw matrices + pipeline to: D:\Mortgage-Credit-Risk-Analytics\Data\features
raw features: 29
